<a href="https://colab.research.google.com/github/rkay0001-Monash/MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches/blob/main/Tree_Prediction_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Important Notice — Dataset Access and Adaptation

## Dataset Accessibility

**This notebook uses a dataset that is not publicly accessible.** The original dataset is stored in a private Google Drive folder, so users will not be able to access the images unless they have been granted permission by the dataset owner. The notebook's dataset-loading process is configured for this specific folder and its file-naming conventions.

Users who do not have access to the original dataset will need to provide their own suitable dataset and modify the relevant parts of the notebook before running the inference workflow.

## Using the Notebook with Other Datasets

The notebook may be adapted to other tree-image datasets, but additional configuration and code changes may be required. In particular, users should consider the following:

- **Dataset access and location:** Update the Google Drive mounting and dataset-loading configuration to point to a folder they can access. Alternatively, replace the Google Drive loading process with another suitable data source.
- **File naming and pairing:** The current implementation identifies seasons and image modalities from filenames, such as `R21U17_Winter_RGB-13-30-50.png` and `R21U17_Winter_D-13-30-50.png`. Other datasets may require changes to the filename parsing, RGB/depth pairing, and Summer/Winter classification logic.
- **RGB and depth image availability:** The U-Net workflow expects paired RGB and depth images. Datasets without depth images will require a different preprocessing approach or modifications to the model input pipeline.
- **Image preprocessing:** The current workflow uses bottom-centred square cropping, resizing, depth normalisation, and depth-based masking of RGB images. These steps may need to be adjusted to match the image dimensions, depth representation, and preprocessing requirements of another dataset.
- **U-Net model compatibility:** The branch segmentation model was developed for a specific task and input format. Using it on another dataset may require changes to the input pipeline, model architecture, trained weights, or output postprocessing. For reliable results on substantially different images, retraining or fine-tuning may be necessary.
- **YOLO11 apple segmentation:** The apple segmentation workflow uses a separately trained or configured YOLO11 model. Users working with different fruit types, objects, or image conditions may need a compatible model and appropriate trained weights.
- **Tree matching and seasonal comparisons:** The Summer/Winter comparison relies on matching tree identifiers across seasons. Users with different identifiers or no seasonal images should modify or disable this comparison step.
- **Dependencies and hardware:** Install the required Python libraries and ensure the selected model weights are available. A Google Colab GPU runtime, preferably an NVIDIA T4 GPU when available, is recommended to reduce inference time.

**Important:** Changing the dataset path alone may not be sufficient to make the notebook work with another dataset. The filenames, image modalities, preprocessing pipeline, model inputs, and trained weights must be compatible with the new data. Predictions on a different dataset should be validated before being treated as reliable results.

## 1. Locate project and dataset

In [ ]:
# ============================================================
# IMPORTS, GOOGLE DRIVE ACCESS, AND OPTIONAL DEPENDENCIES
# ============================================================

import importlib.util
import io
import random
import re
import shutil
import subprocess
import sys
import zipfile
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Patch

# Install Ultralytics before importing YOLO if it is not available.
if importlib.util.find_spec("ultralytics") is None:
    print("Installing Ultralytics...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "ultralytics"
    ])
from ultralytics import YOLO

# Mount Google Drive. The dataset shortcut is accessed through the mounted
# filesystem, avoiding thousands of individual Google Drive API downloads.
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook is configured for Google Colab. Open it in Colab to mount Drive."
    ) from exc

drive.mount("/content/drive", force_remount=False)

# Project and source dataset locations in mounted Google Drive.
PROJECT_DIR = Path("/content/tree_prediction_project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/MMA3001 - Numerical Methods and Machine Learning"
)
DRIVE_DATASET_DIR = DRIVE_PROJECT_DIR / "Dataset"

# A local ZIP is created from the mounted Dataset folder, then extracted to
# Colab storage. This avoids Drive API media-download requests for every image.
LOCAL_ZIP_PATH = Path("/content/Dataset.zip")
DATASET_DIR = Path("/content/tree_prediction_dataset_unsorted")
CACHE_MARKER = DATASET_DIR / ".dataset_extracted_ok"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

if not DRIVE_DATASET_DIR.is_dir():
    raise FileNotFoundError(
        "Could not find the Dataset folder through the mounted Google Drive.\n"
        f"Expected folder/shortcut at: {DRIVE_DATASET_DIR}\n\n"
        "Make sure Google Drive is mounted and the Dataset shortcut is inside "
        "'My Drive/MMA3001 - Numerical Methods and Machine Learning'. If the "
        "shortcut does not appear as a folder in Colab, open the shortcut in "
        "Google Drive and use Organize > Add shortcut to place it in that folder."
    )

# Reuse the extracted dataset on reruns in the same Colab runtime.
def count_dataset_images(folder):
    return sum(
        1 for item in Path(folder).rglob("*")
        if item.is_file() and item.suffix.lower() in IMAGE_EXTENSIONS
    )

cached_image_count = count_dataset_images(DATASET_DIR) if CACHE_MARKER.is_file() else 0

if cached_image_count:
    print(f"Using cached dataset: {cached_image_count} images in {DATASET_DIR}")
else:
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    if LOCAL_ZIP_PATH.exists():
        LOCAL_ZIP_PATH.unlink()

    print(f"Creating a local ZIP from the mounted Drive folder: {DRIVE_DATASET_DIR}")
    print("This reads the source images through the Drive mount; it does not use the Drive API to download each image.")

    source_image_count = count_dataset_images(DRIVE_DATASET_DIR)
    if source_image_count == 0:
        raise RuntimeError(
            f"The Dataset folder was found but contains no supported images: {DRIVE_DATASET_DIR}"
        )
    print(f"Found {source_image_count} source images. Creating ZIP...")

    # Archive the contents of Dataset (not the outer Dataset directory).
    # The archive is temporary and is created only in Colab's local storage.
    with zipfile.ZipFile(LOCAL_ZIP_PATH, mode="w", compression=zipfile.ZIP_STORED) as archive:
        for source_file in DRIVE_DATASET_DIR.rglob("*"):
            if source_file.is_file():
                archive.write(source_file, arcname=source_file.relative_to(DRIVE_DATASET_DIR))

    print(f"Local ZIP created: {LOCAL_ZIP_PATH.stat().st_size / (1024**3):.2f} GiB. Extracting...")
    temp_extract_dir = Path("/content/tree_prediction_dataset_extract_tmp")
    if temp_extract_dir.exists():
        shutil.rmtree(temp_extract_dir)
    temp_extract_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as archive:
            members = archive.infolist()
            root = temp_extract_dir.resolve()
            for item in members:
                target = (temp_extract_dir / item.filename).resolve()
                if target != root and root not in target.parents:
                    raise RuntimeError(f"Unsafe path in ZIP archive: {item.filename!r}")
            archive.extractall(temp_extract_dir)

        extracted_image_count = count_dataset_images(temp_extract_dir)
        if extracted_image_count == 0:
            raise RuntimeError("The Dataset folder was zipped and extracted, but no supported images were found.")

        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        temp_extract_dir.rename(DATASET_DIR)
        (DATASET_DIR / ".dataset_extracted_ok").write_text(
            f"images={extracted_image_count}\n", encoding="utf-8"
        )
        print(f"Extracted {extracted_image_count} images to {DATASET_DIR}")
    except Exception:
        if temp_extract_dir.exists():
            shutil.rmtree(temp_extract_dir, ignore_errors=True)
        raise
    finally:
        # Keep only the extracted dataset; remove the temporary ZIP to save space.
        if LOCAL_ZIP_PATH.exists():
            LOCAL_ZIP_PATH.unlink()

print("Dataset staging complete. Images will be paired by filename in the next cell.")


## 2. Download model code and trained checkpoint

In [ ]:
REPO_URL = (
    "https://github.com/rkay0001-Monash/"
    "MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches/"
    "archive/refs/heads/main.zip"
)

REPO_DIR = (
    PROJECT_DIR
    / "MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches"
)

WEIGHTS_URL = (
    "https://github.com/rkay0001-Monash/"
    "MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches/"
    "releases/download/v1.0.0/trained_weights.pth"
)

WEIGHTS_DIR = PROJECT_DIR / "weights"
WEIGHTS_PATH = WEIGHTS_DIR / "trained_weights.pth"

# Download and extract the repository if it is not already present.
if not REPO_DIR.exists():
    print("Downloading repository ZIP...")
    try:
        with urllib.request.urlopen(REPO_URL) as response:
            zip_data = response.read()
    except Exception as exc:
        raise RuntimeError(
            "Could not download the U-Net repository. Check that the GitHub "
            f"repository URL is valid and accessible: {REPO_URL}\nOriginal error: {exc}"
        ) from exc

    with zipfile.ZipFile(io.BytesIO(zip_data)) as archive:
        members = archive.namelist()
        root_folder = members[0].split("/")[0]

        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

        for member in members:
            relative_path = member[len(root_folder) + 1:]
            if not relative_path:
                continue

            destination = REPO_DIR / relative_path

            if member.endswith("/"):
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, open(destination, "wb") as target:
                    target.write(source.read())

    print("Repository downloaded successfully.")
else:
    print("Using existing repository:", REPO_DIR)

# Download the trained model weights if needed.
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

if not WEIGHTS_PATH.exists() or WEIGHTS_PATH.stat().st_size == 0:
    print("Downloading checkpoint (about 355 MB)...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

print(f"Checkpoint size: {WEIGHTS_PATH.stat().st_size / (1024**2):.1f} MB")
print("Repository path:", REPO_DIR)
print("Checkpoint path:", WEIGHTS_PATH)


## 3. Load the U-Net model

In [ ]:
# ============================================================
# CELL 3: IMPORT U-Net MODEL AND LOAD TRAINED WEIGHTS
# ============================================================
# ------------------------------------------------------------
# 1. Verify repository and checkpoint paths
# ------------------------------------------------------------

REPO_DIR = Path(REPO_DIR).resolve()
WEIGHTS_PATH = Path(WEIGHTS_PATH).resolve()

if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Repository folder not found:\n{REPO_DIR}"
    )

if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"Trained weights not found:\n{WEIGHTS_PATH}"
    )

# ------------------------------------------------------------
# 2. Import the UNet architecture
# ------------------------------------------------------------

# Add the repository root to Python's import path.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# The repository contains:
# unet/unet_model.py
# unet/unet_parts.py
# unet/__init__.py

UNet = importlib.import_module("unet").UNet

print("UNet architecture imported successfully.")

# ------------------------------------------------------------
# 3. Select CPU or GPU
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA GPU unavailable; using CPU.")

# ------------------------------------------------------------
# 4. Define model settings
# ------------------------------------------------------------

INPUT_SIZE = 256
OUTPUT_SIZE = 480
THRESHOLD = 0.5

# ------------------------------------------------------------
# 5. Initialise the model architecture
# ------------------------------------------------------------

model = UNet(
    n_channels=3,
    n_classes=1,
    bilinear=False
)

# ------------------------------------------------------------
# 6. Load trained checkpoint
# ------------------------------------------------------------

print("Loading trained checkpoint...")

try:
    checkpoint = torch.load(
        WEIGHTS_PATH,
        map_location=DEVICE,
        weights_only=True
    )
except TypeError:
    # Compatibility with older PyTorch versions.
    checkpoint = torch.load(
        WEIGHTS_PATH,
        map_location=DEVICE
    )

# The checkpoint may contain the state dictionary directly,
# or inside a dictionary under "model_state_dict".
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

print("Model loaded successfully.")
print(f"Input size: {INPUT_SIZE} x {INPUT_SIZE}")
print(f"Output mask size: {OUTPUT_SIZE} x {OUTPUT_SIZE}")
print(f"Probability threshold: {THRESHOLD}")


## 4. Match RGB and D images by filename stem

In [ ]:
# ============================================================
# MATCH RGB / DEPTH PAIRS FROM THE UNSORTED DRIVE DATASET
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}


def list_images(folder):
    """Return supported image files recursively, sorted by path."""
    folder = Path(folder)
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def classify_filename(path):
    """Return (season, modality) from names like R21U17_Winter_RGB-13-30-50.png."""
    match = re.search(r"_(Summer|Winter)_(RGB|D)-", Path(path).stem, flags=re.IGNORECASE)
    if not match:
        return None, None
    return match.group(1).title(), match.group(2).upper()


def normalise_stem(path):
    """Normalise RGB and depth filename stems so paired files share a key."""
    stem = Path(path).stem
    stem = re.sub(r"_(RGB|D)-", "_", stem, flags=re.IGNORECASE)
    return stem.casefold()


def stem_index(paths):
    """Index paths by normalised stem; return the index and duplicate paths."""
    index = {}
    duplicates = []
    for path in sorted(paths):
        key = normalise_stem(path)
        if key in index:
            duplicates.append(path)
        else:
            index[key] = path
    return index, duplicates


all_images = list_images(DATASET_DIR)
classified = {"Summer": {"RGB": [], "D": []}, "Winter": {"RGB": [], "D": []}}
unclassified = []
for path in all_images:
    season, modality = classify_filename(path)
    if season is None:
        unclassified.append(path)
    else:
        classified[season][modality].append(path)

print(f"Total supported images found: {len(all_images)}")
print(f"Files not matching the expected season/modality filename pattern: {len(unclassified)}")
if unclassified:
    print("Examples of unclassified files:", [p.name for p in unclassified[:5]])

PAIRS = {}
for season in ("Summer", "Winter"):
    rgb_index, rgb_duplicates = stem_index(classified[season]["RGB"])
    depth_index, depth_duplicates = stem_index(classified[season]["D"])
    common_keys = sorted(rgb_index.keys() & depth_index.keys())
    PAIRS[season] = [(rgb_index[key], depth_index[key]) for key in common_keys]

    print(f"\n{season}:")
    print(f"  RGB images found: {len(rgb_index)}")
    print(f"  Depth images found: {len(depth_index)}")
    print(f"  Matched RGB/depth pairs: {len(PAIRS[season])}")
    if rgb_duplicates or depth_duplicates:
        print(
            f"  WARNING: duplicate normalised keys — RGB={len(rgb_duplicates)}, "
            f"depth={len(depth_duplicates)}"
        )
    unmatched_rgb = rgb_index.keys() - depth_index.keys()
    unmatched_depth = depth_index.keys() - rgb_index.keys()
    if unmatched_rgb:
        print(f"  Unmatched RGB images: {len(unmatched_rgb)}")
    if unmatched_depth:
        print(f"  Unmatched depth images: {len(unmatched_depth)}")

WINTER_PAIRS = PAIRS["Winter"]
WINTER_RGB_IMAGES = [rgb_path for rgb_path, _ in WINTER_PAIRS]

if not PAIRS["Summer"]:
    raise RuntimeError(
        "No matching Summer RGB/depth pairs found. Expected filenames like "
        "R21U17_Summer_RGB-14-26-37.png and R21U17_Summer_D-14-26-37.png."
    )
if not WINTER_PAIRS:
    raise RuntimeError(
        "No matching Winter RGB/depth pairs found. Expected filenames like "
        "R21U17_Winter_RGB-13-30-50.png and R21U17_Winter_D-13-30-50.png."
    )

print("\nUnsorted dataset pairing completed successfully.")


## 5. Preprocess and predict
The depth image is combined with RGB through soft masking, as in the established inference workflow. It is not a fourth model input channel.

In [ ]:
# ============================================================
# PREPROCESSING AND U-Net INFERENCE
# ============================================================


def bottom_center_square_crop(image: np.ndarray) -> np.ndarray:
    """Return a square crop centred horizontally and aligned to the bottom.

    The crop geometry is applied identically to RGB and depth images so their
    pixels stay aligned. For a 640×480 landscape image, this keeps columns
    80–559 and all 480 rows. For portrait images, the crop is bottom-aligned.

    Args:
        image: A 2-D or 3-D NumPy image array.

    Returns:
        A copy of the largest square crop contained in ``image``.

    Raises:
        ValueError: If ``image`` is empty or has fewer than two dimensions.
    """
    if image is None or image.ndim < 2 or image.shape[0] == 0 or image.shape[1] == 0:
        raise ValueError("Cannot crop an empty image.")

    height, width = image.shape[:2]
    side = min(height, width)
    x_start = (width - side) // 2
    y_start = height - side  # Bottom alignment; zero for landscape images.
    return image[y_start:y_start + side, x_start:x_start + side].copy()


def preprocess_pair(rgb_path: Path, d_path: Path):
    """Load and preprocess a paired RGB/depth image for U-Net and YOLO.

    RGB and depth images are checked for matching dimensions, then cropped
    using the same geometry. U-Net receives depth-masked RGB at ``INPUT_SIZE``;
    YOLO and display functions use the unmasked RGB reference at
    ``OUTPUT_SIZE``.

    Args:
        rgb_path: Path to the RGB image.
        d_path: Path to the matching grayscale depth image.

    Returns:
        Tuple containing the resized RGB image, normalised depth image,
        depth-masked RGB image, batched U-Net tensor, and cropped RGB reference.

    Raises:
        ValueError: If an image cannot be read, image dimensions differ, or
            the depth image contains no non-zero pixels.
    """
    bgr = cv2.imread(str(rgb_path), cv2.IMREAD_COLOR)
    depth = cv2.imread(str(d_path), cv2.IMREAD_GRAYSCALE)

    if bgr is None:
        raise ValueError(f"Could not read RGB image: {rgb_path}")
    if depth is None:
        raise ValueError(f"Could not read depth image: {d_path}")
    if bgr.shape[:2] != depth.shape[:2]:
        raise ValueError(
            f"RGB/depth dimensions do not match for {rgb_path.name}: "
            f"RGB={bgr.shape[:2]}, depth={depth.shape[:2]}"
        )
    if not np.any(depth):
        raise ValueError(f"Depth image contains only zeros: {d_path}")

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Crop both images before resizing to preserve pixel alignment.
    rgb_crop = bottom_center_square_crop(rgb)
    depth_crop = bottom_center_square_crop(depth)

    # U-Net input resolution: area interpolation for RGB, nearest for depth.
    rgb256 = cv2.resize(
        rgb_crop, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_AREA
    )
    depth256 = cv2.resize(
        depth_crop, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_NEAREST
    )

    # Scale depth to [0, 1] and use it as a soft per-pixel RGB mask.
    depth_norm = depth256.astype(np.float32)
    depth_norm /= float(depth_norm.max())
    depth_masked_rgb = np.clip(
        rgb256.astype(np.float32) * depth_norm[..., None], 0, 255
    ).astype(np.uint8)

    # U-Net expects a normalised CHW tensor with a batch dimension.
    input_tensor = torch.from_numpy(
        (depth_masked_rgb.astype(np.float32) / 255.0).transpose(2, 0, 1)
    ).unsqueeze(0)

    # Keep an unmasked RGB reference for apple detection and visualisation.
    rgb480 = cv2.resize(
        rgb_crop, (OUTPUT_SIZE, OUTPUT_SIZE), interpolation=cv2.INTER_AREA
    )

    return rgb256, depth_norm, depth_masked_rgb, input_tensor, rgb480


def predict_pair(rgb_path: Path, d_path: Path):
    """Predict the U-Net branch probability map, binary mask, and overlay.

    The model output is treated as logits and passed through a sigmoid.
    The probability map is resized with linear interpolation; thresholding is
    performed after resizing to preserve the configured output resolution.

    Args:
        rgb_path: Path to the RGB image.
        d_path: Path to the matching depth image.

    Returns:
        ``(probability_480, mask_480, overlay_480, rgb480)``. The binary mask
        uses 0 for background and 255 for predicted branches.
    """
    _, _, _, input_tensor, rgb480 = preprocess_pair(rgb_path, d_path)

    with torch.inference_mode():
        logits = model(input_tensor.to(DEVICE))
        probabilities256 = torch.sigmoid(logits)[0, 0].cpu().numpy()

    probabilities480 = cv2.resize(
        probabilities256,
        (OUTPUT_SIZE, OUTPUT_SIZE),
        interpolation=cv2.INTER_LINEAR,
    )
    mask480 = (probabilities480 >= THRESHOLD).astype(np.uint8) * 255

    # Blend branch pixels with red while retaining some RGB detail.
    overlay = rgb480.copy()
    selected = mask480 > 0
    red = np.zeros_like(overlay)
    red[..., 0] = 255
    alpha = 0.45
    overlay[selected] = (
        (1 - alpha) * overlay[selected].astype(np.float32)
        + alpha * red[selected].astype(np.float32)
    ).clip(0, 255).astype(np.uint8)

    return probabilities480, mask480, overlay, rgb480


## 6. Run Summer U-Net predictions in memory

RGB/depth pairs receive the same bottom-centred square crop. U-Net uses a 256 × 256 depth-masked RGB input; branch probabilities and masks are mapped to 480 × 480 for comparison with YOLO. Predictions remain in memory and are not written to disk.


In [ ]:
# ============================================================
# RUN SUMMER U-Net PREDICTIONS IN MEMORY — NO OUTPUT FILES
# ============================================================

ERRORS = []
RESULTS = {"Summer": []}

summer_pairs = PAIRS["Summer"]
print(f"\nProcessing Summer U-Net predictions: {len(summer_pairs)} matched pairs")

for i, (rgb_path, depth_path) in enumerate(summer_pairs, start=1):
    try:
        probability, mask, overlay, rgb480 = predict_pair(rgb_path, depth_path)
        RESULTS["Summer"].append({
            "stem": rgb_path.stem,
            "probability": probability,
            "mask": mask,
            "overlay": overlay,
            "rgb480": rgb480,
            "rgb_path": rgb_path,
            "d_path": depth_path,
        })
        if i == 1 or i % 10 == 0 or i == len(summer_pairs):
            print(f"  {i}/{len(summer_pairs)} complete")
    except Exception as exc:
        ERRORS.append({
            "season": "Summer",
            "rgb": str(rgb_path),
            "depth": str(depth_path),
            "error": repr(exc),
        })
        print(f"  SKIPPED {rgb_path.name}: {exc}")

print("\nSummer U-Net predictions:", len(RESULTS["Summer"]))
print("Failed Summer pairs:", len(ERRORS))
print("Summer predictions are held in memory only; no results were saved locally.")


## 7. Compare Summer and Winter branch predictions for the same tree IDs

The left and right columns show U-Net branch predictions from Summer and Winter images that share the same tree ID (the leading `RxxUxx` portion of each filename). One view is selected per season for each tree ID, so the figure contains up to five unique trees rather than ten unrelated trees. Winter predictions are generated on demand for the selected matched pairs only; YOLO apple detection remains Summer-only. If fewer than five tree IDs exist in both seasons, all shared IDs are displayed.


In [ ]:
# ============================================================
# COMPARE SUMMER AND WINTER BRANCH PREDICTIONS BY TREE ID
# ============================================================
def extract_tree_id(path):
    """Extract the shared tree identifier (e.g. ``R05U19``) from a filename.

    The comparison assumes the leading R-number/U-number token identifies a
    tree across seasons. Filenames that do not begin with this pattern return
    None and are excluded from same-tree matching.
    """
    match = re.match(r"^(R\d+U\d+)", Path(path).stem, flags=re.IGNORECASE)
    return match.group(1).upper() if match else None


def show_summer_branches_with_winter_predictions(n=5, seed=None):
    """Display paired Summer/Winter U-Net predictions for the same tree IDs.

    Summer predictions are reused from ``RESULTS['Summer']``. Winter RGB/depth
    pairs are inferred on demand. Each displayed row represents one unique
    tree ID shared by both seasons; one image/view is randomly selected from
    each season for that ID. This avoids comparing unrelated trees.

    Args:
        n: Maximum number of unique tree IDs to display (default: five).
        seed: Optional seed for repeatable selection of tree IDs and views.

    Notes:
        The function does not save prediction images. The same tree ID does not
        guarantee the camera pose or exact branch geometry is identical between
        seasons; it only pairs samples by the dataset's filename convention.
    """
    if n < 1:
        raise ValueError("n must be at least 1.")

    summer_results = RESULTS.get("Summer", [])
    winter_pairs = list(WINTER_PAIRS)

    if not summer_results:
        print("No successful Summer branch predictions are available to display.")
        return
    if not winter_pairs:
        print("No matched Winter RGB/depth pairs are available for U-Net inference.")
        return

    # Group Summer prediction results and Winter RGB/depth pairs by tree ID.
    summer_by_tree = {}
    for result in summer_results:
        tree_id = extract_tree_id(result["rgb_path"])
        if tree_id is not None:
            summer_by_tree.setdefault(tree_id, []).append(result)

    winter_by_tree = {}
    for rgb_path, depth_path in winter_pairs:
        tree_id = extract_tree_id(rgb_path)
        if tree_id is not None:
            winter_by_tree.setdefault(tree_id, []).append((rgb_path, depth_path))

    shared_tree_ids = sorted(set(summer_by_tree) & set(winter_by_tree))
    if not shared_tree_ids:
        print(
            "No shared tree IDs were found between Summer predictions and Winter pairs. "
            "Check that filenames begin with the same RxxUxx tree ID in both seasons."
        )
        print("Example Summer IDs:", sorted(summer_by_tree)[:10])
        print("Example Winter IDs:", sorted(winter_by_tree)[:10])
        return

    rng = random.Random(seed)
    selected_tree_ids = rng.sample(shared_tree_ids, min(n, len(shared_tree_ids)))
    print(f"Shared tree IDs available: {len(shared_tree_ids)}")
    print(f"Unique trees selected for comparison: {len(selected_tree_ids)}")

    # Select one Summer view and one Winter view for each selected tree.
    comparison_rows = []
    winter_errors = []
    for tree_id in selected_tree_ids:
        summer_item = rng.choice(summer_by_tree[tree_id])
        winter_rgb_path, winter_depth_path = rng.choice(winter_by_tree[tree_id])

        try:
            probability, mask, overlay, rgb480 = predict_pair(
                winter_rgb_path, winter_depth_path
            )
            winter_item = {
                "stem": winter_rgb_path.stem,
                "probability": probability,
                "mask": mask,
                "overlay": overlay,
                "rgb480": rgb480,
                "rgb_path": winter_rgb_path,
                "d_path": winter_depth_path,
            }
            comparison_rows.append((tree_id, summer_item, winter_item))
        except Exception as exc:
            winter_errors.append((winter_rgb_path.name, repr(exc)))
            print(f"WARNING: skipped Winter pair {winter_rgb_path.name}: {exc}")

    if winter_errors:
        print(f"Winter prediction failures: {len(winter_errors)}")
    if not comparison_rows:
        print("No valid same-tree Summer/Winter predictions could be displayed.")
        return

    row_count = len(comparison_rows)
    fig, axes = plt.subplots(
        nrows=row_count,
        ncols=2,
        figsize=(12, 5.0 * row_count),
        squeeze=False,
    )
    fig.suptitle(
        "U-Net Branch Predictions: Same Trees in Summer vs Winter",
        fontsize=15,
        y=0.99,
    )

    for row, (tree_id, summer_item, winter_item) in enumerate(comparison_rows):
        axes[row, 0].imshow(summer_item["overlay"])
        axes[row, 0].set_title(
            f"Summer — {tree_id}\n{summer_item['stem']}",
            fontsize=9,
            pad=8,
        )
        axes[row, 0].axis("off")

        axes[row, 1].imshow(winter_item["overlay"])
        axes[row, 1].set_title(
            f"Winter — {tree_id}\n{winter_item['stem']}",
            fontsize=9,
            pad=8,
        )
        axes[row, 1].axis("off")

    fig.tight_layout(rect=(0, 0, 1, 0.92), h_pad=2.4, w_pad=0.8)
    plt.show()


show_summer_branches_with_winter_predictions(n=5, seed=None)


## 8. Detect Summer apples with YOLO11 segmentation

This stage processes the Summer RGB references generated by the U-Net preprocessing pipeline. It uses the COCO-pretrained YOLO11 segmentation checkpoint and filters predictions to the `apple` class. `YOLO_CONFIDENCE` is the minimum detection confidence; lowering it may find more difficult apples but can increase false positives. Predictions are kept in memory only.


In [ ]:
# ============================================================
# YOLO11 APPLE MASKS — SUMMER ONLY, IN MEMORY ONLY
# ============================================================
YOLO_MODEL_PATH = "yolo11n-seg.pt"
YOLO_CONFIDENCE = 0.1
YOLO_IMAGE_SIZE = OUTPUT_SIZE  # model input source and output masks are 480x480

print("Loading YOLO11 segmentation model...")
apple_model = YOLO(YOLO_MODEL_PATH)

names = apple_model.names
if isinstance(names, dict):
    apple_class_ids = [int(k) for k, v in names.items() if str(v).lower() == "apple"]
else:
    apple_class_ids = [i for i, name in enumerate(names) if str(name).lower() == "apple"]

if not apple_class_ids:
    raise RuntimeError(
        "The loaded YOLO model does not include an 'apple' class. "
        "Please verify the checkpoint is the COCO-trained yolo11n-seg.pt model."
    )
APPLE_CLASS_ID = apple_class_ids[0]

SUMMER_RESULTS = RESULTS.get("Summer", [])
if not SUMMER_RESULTS:
    raise RuntimeError(
        "No successful Summer branch predictions are available. "
        "Run the U-Net prediction cell first."
    )

APPLE_RESULTS = []
APPLE_ERRORS = []
yolo_device = 0 if torch.cuda.is_available() else "cpu"
print(f"Apple class ID: {APPLE_CLASS_ID}; device: {yolo_device}")
print(f"Processing {len(SUMMER_RESULTS)} Summer images only...")

for i, item in enumerate(SUMMER_RESULTS, start=1):
    stem = item["stem"]
    rgb480 = item["rgb480"]  # square-cropped RGB image, exactly 480x480

    try:
        input_bgr = cv2.cvtColor(rgb480, cv2.COLOR_RGB2BGR)
        prediction = apple_model.predict(
            source=input_bgr,
            imgsz=YOLO_IMAGE_SIZE,
            conf=YOLO_CONFIDENCE,
            classes=[APPLE_CLASS_ID],
            retina_masks=True,
            device=yolo_device,
            verbose=False
        )[0]

        apple_mask = np.zeros((OUTPUT_SIZE, OUTPUT_SIZE), dtype=np.uint8)
        instance_count = 0

        if prediction.masks is not None and prediction.boxes is not None:
            masks = prediction.masks.data.cpu().numpy()
            class_ids = prediction.boxes.cls.cpu().numpy().astype(int)
            for instance_mask, class_id in zip(masks, class_ids):
                if class_id != APPLE_CLASS_ID:
                    continue
                if instance_mask.shape != (OUTPUT_SIZE, OUTPUT_SIZE):
                    instance_mask = cv2.resize(
                        instance_mask.astype(np.float32),
                        (OUTPUT_SIZE, OUTPUT_SIZE),
                        interpolation=cv2.INTER_LINEAR
                    )
                apple_mask[instance_mask > 0.5] = 255
                instance_count += 1

        apple_overlay = rgb480.copy()
        apple_pixels = apple_mask > 0
        green = np.zeros_like(apple_overlay)
        green[..., 1] = 255
        alpha = 0.45
        apple_overlay[apple_pixels] = (
            (1 - alpha) * apple_overlay[apple_pixels].astype(np.float32)
            + alpha * green[apple_pixels].astype(np.float32)
        ).clip(0, 255).astype(np.uint8)

        APPLE_RESULTS.append({
            "stem": stem,
            "rgb480": rgb480,
            "apple_mask": apple_mask,
            "apple_overlay": apple_overlay,
            "apple_count": instance_count,
            "branch_mask": item["mask"],
            "rgb_path": item["rgb_path"]
        })

        if i == 1 or i % 25 == 0 or i == len(SUMMER_RESULTS):
            print(f"  {i}/{len(SUMMER_RESULTS)} processed")

    except Exception as exc:
        APPLE_ERRORS.append({"stem": stem, "error": repr(exc)})
        print(f"  SKIPPED {stem}: {exc}")

print("\nSummer apple segmentation finished.")
print("Images processed successfully:", len(APPLE_RESULTS))
print("Images with at least one apple:", sum(r["apple_count"] > 0 for r in APPLE_RESULTS))
print("Total apple instances detected:", sum(r["apple_count"] for r in APPLE_RESULTS))
print("Apple masks and overlays are held in memory only; nothing was saved locally.")


## 9. Display 5 random Summer apple-segmentation samples

Each row shows the same randomly selected Summer image in three columns: pure cropped RGB, binary apple mask, and the apple mask overlaid on RGB. A fresh sample is selected whenever this cell is rerun.

In [ ]:
# ============================================================
# DISPLAY RANDOM SUMMER APPLE MASKS — THREE COLUMNS
# ============================================================

if not APPLE_RESULTS:
    print("No successful Summer apple-segmentation results to display.")
else:
    chosen_apple = random.sample(APPLE_RESULTS, min(5, len(APPLE_RESULTS)))
    row_count = len(chosen_apple)
    fig, axes = plt.subplots(
        row_count, 3, figsize=(16, 5.0 * row_count), squeeze=False
    )
    fig.suptitle(
        "Random Summer YOLO11 apple-segmentation samples",
        fontsize=15,
        y=0.995,
    )

    for row, item in enumerate(chosen_apple):
        # Column 1: pure RGB reference, with no detections drawn on it.
        axes[row, 0].imshow(item["rgb480"])
        axes[row, 0].set_title(f'{item["stem"]} — pure RGB', fontsize=9, pad=8)
        axes[row, 0].axis("off")

        # Column 2: binary mask (white = detected apple, black = background).
        axes[row, 1].imshow(item["apple_mask"], cmap="gray", vmin=0, vmax=255)
        axes[row, 1].set_title(
            f'Binary apple mask ({item["apple_count"]} instances)',
            fontsize=9, pad=8
        )
        axes[row, 1].axis("off")

        # Column 3: green apple mask blended over the same RGB image.
        axes[row, 2].imshow(item["apple_overlay"])
        axes[row, 2].set_title("RGB + apple mask overlay", fontsize=9, pad=8)
        axes[row, 2].axis("off")

    fig.tight_layout(rect=(0, 0, 1, 0.92), h_pad=2.4, w_pad=0.7)
    plt.show()


## 10. Combine branch and apple masks

The U-Net branch mask and YOLO apple mask are combined at 480 × 480. The class mask uses `0 = background`, `1 = branch only`, `2 = apple only`, and `3 = overlap`. The visual overlay uses red for branches, green for apples, and yellow where the masks overlap. Results remain in memory only.


In [ ]:
# ============================================================
# COMBINE SUMMER TREE-BRANCH AND APPLE MASKS — IN MEMORY ONLY
# ============================================================

COMBINED_RESULTS = []

for item in APPLE_RESULTS:
    stem = item["stem"]
    rgb480 = item["rgb480"]
    branch_mask = item["branch_mask"]
    apple_mask = item["apple_mask"]

    # Both masks must align exactly with the 480x480 cropped RGB image.
    if branch_mask.shape != (OUTPUT_SIZE, OUTPUT_SIZE):
        branch_mask = cv2.resize(
            branch_mask, (OUTPUT_SIZE, OUTPUT_SIZE),
            interpolation=cv2.INTER_NEAREST
        )
    if apple_mask.shape != (OUTPUT_SIZE, OUTPUT_SIZE):
        apple_mask = cv2.resize(
            apple_mask, (OUTPUT_SIZE, OUTPUT_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

    branch_pixels = branch_mask > 0
    apple_pixels = apple_mask > 0
    overlap_pixels = branch_pixels & apple_pixels

    # Class-index mask: 0 background, 1 branch-only, 2 apple-only, 3 overlap.
    combined_mask = np.zeros((OUTPUT_SIZE, OUTPUT_SIZE), dtype=np.uint8)
    combined_mask[branch_pixels] = 1
    combined_mask[apple_pixels] = 2
    combined_mask[overlap_pixels] = 3

    # RGB overlay: branches red, apples green, overlap yellow.
    colour_layer = np.zeros_like(rgb480)
    colour_layer[branch_pixels] = [255, 0, 0]
    colour_layer[apple_pixels] = [0, 255, 0]
    colour_layer[overlap_pixels] = [255, 255, 0]

    labelled_pixels = branch_pixels | apple_pixels
    combined_overlay = rgb480.copy()
    alpha = 0.50
    combined_overlay[labelled_pixels] = (
        (1 - alpha) * rgb480[labelled_pixels].astype(np.float32)
        + alpha * colour_layer[labelled_pixels].astype(np.float32)
    ).clip(0, 255).astype(np.uint8)

    COMBINED_RESULTS.append({
        "stem": stem,
        "rgb480": rgb480,
        "branch_mask": branch_mask,
        "apple_mask": apple_mask,
        "combined_mask": combined_mask,
        "combined_overlay": combined_overlay,
        "apple_count": item["apple_count"]
    })

print("Combined Summer results:", len(COMBINED_RESULTS))
print("All combined masks and overlays are held in memory only; nothing was saved locally.")
print("Mask labels: 0=background, 1=branch-only, 2=apple-only, 3=overlap")


## 11. Display 5 random combined Summer segmentation results

Each row shows the pure cropped RGB image alongside the combined branch-and-apple overlay. Red = branch-only pixels, green = apple-only pixels, and yellow = overlap. A fresh random subset is selected whenever the cell is rerun.

In [ ]:
# ============================================================
# DISPLAY RANDOM COMBINED SUMMER SEGMENTATION RESULTS
# ============================================================
def show_random_combined_results(results, n=5, seed=None):
    """Display up to ``n`` random RGB/combined-overlay pairs.

    Args:
        results: List of dictionaries created by the combined-mask cell.
        n: Maximum number of examples to display.
        seed: Optional seed for repeatable sample selection.

    Notes:
        This function only displays images; it does not write files.
    """
    if not results:
        print("No combined Summer results are available to display.")
        return
    if n < 1:
        raise ValueError("n must be at least 1.")

    # Fresh random subset on every run unless an explicit seed is supplied.
    if seed is not None:
        chosen = random.Random(seed).sample(results, min(n, len(results)))
    else:
        chosen = random.sample(results, min(n, len(results)))
    row_count = len(chosen)

    # Two columns: cropped RGB reference and combined segmentation overlay.
    fig, axes = plt.subplots(
        nrows=row_count,
        ncols=2,
        figsize=(12, 5.0 * row_count),
        squeeze=False,
    )
    fig.suptitle(
        "Summer: RGB Reference and Combined Branch/Apple Segmentation",
        fontsize=15,
        y=0.995,
    )

    for row, item in enumerate(chosen):
        axes[row, 0].imshow(item["rgb480"])
        axes[row, 0].set_title(
            f'RGB reference — {item["stem"]}', fontsize=10, pad=8
        )
        axes[row, 0].axis("off")

        axes[row, 1].imshow(item["combined_overlay"])
        axes[row, 1].set_title(
            "Combined segmentation overlay", fontsize=10, pad=8
        )
        axes[row, 1].axis("off")

    legend_items = [
        Patch(facecolor="red", label="Branches only"),
        Patch(facecolor="lime", label="Apples only"),
        Patch(facecolor="yellow", label="Overlap"),
    ]

    # Reserve the top for the figure title and the bottom for the legend.
    fig.tight_layout(rect=(0, 0.07, 1, 0.92), h_pad=2.4, w_pad=0.8)
    fig.legend(
        handles=legend_items,
        loc="lower center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 0.015),
    )
    plt.show()


show_random_combined_results(COMBINED_RESULTS, n=5, seed=None)


## Notes and assumptions

- Ground-truth masks are not required for inference.
- Summer and Winter RGB/depth files are classified from filenames and paired by normalised filename stems in the unsorted Drive dataset. Filenames must contain `_<Summer|Winter>_<RGB|D>-`.
- The RGB and depth images must have matching spatial dimensions. Both receive the same bottom-centred square crop.
- Depth images are read as grayscale. Depth is normalised by its maximum and multiplied into RGB as a soft mask; this is a three-channel U-Net input, not a fourth channel.
- U-Net input is `INPUT_SIZE × INPUT_SIZE`; branch probabilities and masks are mapped to `OUTPUT_SIZE × OUTPUT_SIZE`.
- U-Net is run over all matched Summer pairs. The comparison cell finds tree IDs shared across Summer and Winter, then infers only one randomly selected Winter pair per selected shared tree ID.
- YOLO11 runs on the unmasked, square-cropped Summer RGB reference and is filtered to the COCO `apple` class. Winter images are not passed through YOLO.
- The combined Summer class mask uses `0 = background`, `1 = branch only`, `2 = apple only`, and `3 = overlap`.
- Combined Summer overlays use red for branches only, green for apples only, and yellow for overlap. The RGB reference, masks, and overlays are aligned at 480 × 480 by default.
- Prediction arrays and overlays remain in memory; this notebook does not save prediction results locally. Drive images are copied to temporary `/content` storage for inference. Repository/checkpoint downloads and the YOLO model-weight download are separate setup actions and may create files in the Colab runtime.
- The COCO-pretrained YOLO11 model may miss small, distant, or heavily occluded apples. The confidence threshold trades detection recall against false positives.
- Square cropping changes the framing. Make sure this is appropriate for the images and consistent with the model's training/inference preprocessing.
- Winter branch masks are model predictions, not ground truth. The seasonal comparison matches the leading RxxUxx tree ID; selected views may still differ in camera position or timestamp. Prediction quality may be lower if Winter imagery differs from Summer training data.
